# exp010_bertimbau_es

**Notes:** BERTimbau base + early stopping — re-run of best model (exp004) with epochs=10, patience=2, max_length=256; tests whether more epochs + ES beats 5-epoch fixed run

Auto-generated Kaggle submission notebook for a transformer experiment.
Trains from scratch on Kaggle GPU and writes `submission.csv`.

## Setup checklist
1. **Accelerator**: GPU T4×2 (or P100) — *Notebook settings → Accelerator*
2. **Internet**: Off at submission time
3. **Dataset**: attach `spr-2026-mammography-report-classification` competition data
4. **Model weights**: attach `neuralmind/bert-base-portuguese-cased` via *Add-ons → Models*
   - Search the Kaggle Models hub for the model name
   - Update `PRETRAINED_PATH` in the config cell below to the mount path
   - Local HuggingFace ID works when internet is ON (testing only)


In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40', 'accelerate>=0.27', 'sentencepiece>=0.1.99',
    'pyyaml>=6.0', 'tqdm>=4.65', 'scikit-learn>=1.3',
    'pandas>=2.0', 'numpy>=1.24',
])


In [ ]:
import os
os.makedirs('src/models', exist_ok=True)
os.makedirs('configs', exist_ok=True)
os.makedirs('experiments', exist_ok=True)


In [ ]:
open('src/__init__.py', 'w').close()  # empty init file


In [ ]:
%%writefile src/data.py
"""
Data loading and fold creation.
Handles both local and Kaggle runtime paths automatically.
"""
import os
from pathlib import Path

import pandas as pd
from sklearn.model_selection import StratifiedKFold

# Searched in order - first match wins
_DATA_DIRS = [
    "/kaggle/input/competitions/spr-2026-mammography-report-classification",
    "/kaggle/input/spr-2026-mammography-report-classification",
    "./spr-2026-mammography-report-classification",
    "../spr-2026-mammography-report-classification",
    "../../spr-2026-mammography-report-classification",
]


def find_data_dir() -> str:
    env_data_dir = os.getenv("SPR2026_DATA_DIR")
    if env_data_dir:
        env_path = Path(env_data_dir)
        if (env_path / "train.csv").exists():
            return str(env_path)

    tried = []
    for data_dir in _DATA_DIRS:
        train_path = Path(data_dir) / "train.csv"
        tried.append(str(train_path))
        if train_path.exists():
            return str(Path(data_dir))

    raise FileNotFoundError(
        "Could not find data directory. Checked these train.csv paths:\n"
        + "\n".join(f"- {path}" for path in tried)
        + "\nSet SPR2026_DATA_DIR or place the competition data in "
        "./spr-2026-mammography-report-classification/."
    )


def load_train(data_dir: str = None) -> pd.DataFrame:
    if data_dir is None:
        data_dir = find_data_dir()
    df = pd.read_csv(Path(data_dir) / "train.csv")
    df.columns = df.columns.str.strip()
    df["target"] = df["target"].astype(int)
    return df


def load_test(data_dir: str = None) -> pd.DataFrame:
    if data_dir is None:
        data_dir = find_data_dir()
    path = Path(data_dir) / "test.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"test.csv not found at {path}.\n"
            "This is expected locally - it only exists in the Kaggle evaluation runtime."
        )
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    return df


def make_folds(df: pd.DataFrame, n_folds: int = 5, seed: int = 42) -> pd.DataFrame:
    """Add a 'fold' column (0..n_folds-1) using stratified split on target."""
    df = df.copy().reset_index(drop=True)
    df["fold"] = -1
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    for fold, (_, val_idx) in enumerate(skf.split(df, df["target"])):
        df.loc[val_idx, "fold"] = fold
    return df


In [ ]:
%%writefile src/features.py
"""
Text preprocessing and TF-IDF feature extraction for Portuguese medical reports.
"""
import pickle
import re
import os
from typing import List, Tuple, Optional

import numpy as np
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer


def clean_text(text: str) -> str:
    """
    Minimal cleaning that preserves medically meaningful tokens.
    - Keeps <DATA> placeholder (correlates with comparative exams → BI-RADS 2/3)
    - Normalises whitespace; lowercases
    - Does NOT strip accents — Portuguese accents carry lexical meaning
    """
    if not isinstance(text, str):
        text = str(text)
    # Collapse newlines / tabs to a single space
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r" {2,}", " ", text).strip()
    text = text.lower()
    return text


def _make_vectorizer(cfg: dict) -> TfidfVectorizer:
    return TfidfVectorizer(
        analyzer=cfg.get("analyzer", "word"),
        ngram_range=tuple(cfg.get("ngram_range", [1, 2])),
        max_features=cfg.get("max_features", 150_000),
        sublinear_tf=cfg.get("sublinear_tf", True),
        min_df=cfg.get("min_df", 2),
        strip_accents=None,  # preserve Portuguese characters
    )


def build_features(
    texts_train: List[str],
    texts_val: Optional[List[str]] = None,
    config: Optional[dict] = None,
) -> Tuple:
    """
    Fit TF-IDF (word + char) on texts_train.
    Returns (X_train, vectorizers) or (X_train, X_val, vectorizers).
    """
    cfg = config or {}
    word_cfg = cfg.get("word_tfidf", {})
    char_cfg  = cfg.get("char_tfidf", {})

    # Default char config if not provided
    if "analyzer" not in char_cfg:
        char_cfg = {"analyzer": "char_wb", "ngram_range": [2, 5],
                    "max_features": 150_000, "sublinear_tf": True, "min_df": 3}

    word_vec = _make_vectorizer(word_cfg)
    char_vec  = _make_vectorizer(char_cfg)

    X_word_train = word_vec.fit_transform(texts_train)
    X_char_train  = char_vec.fit_transform(texts_train)
    X_train = hstack([X_word_train, X_char_train], format="csr")

    vectorizers = (word_vec, char_vec)

    if texts_val is not None:
        X_word_val = word_vec.transform(texts_val)
        X_char_val  = char_vec.transform(texts_val)
        X_val = hstack([X_word_val, X_char_val], format="csr")
        return X_train, X_val, vectorizers

    return X_train, vectorizers


def transform_features(texts: List[str], vectorizers: Tuple) -> csr_matrix:
    word_vec, char_vec = vectorizers
    X_word = word_vec.transform(texts)
    X_char  = char_vec.transform(texts)
    return hstack([X_word, X_char], format="csr")


def save_vectorizers(vectorizers: Tuple, out_dir: str) -> None:
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, "vectorizers.pkl"), "wb") as f:
        pickle.dump(vectorizers, f)


def load_vectorizers(out_dir: str) -> Tuple:
    with open(os.path.join(out_dir, "vectorizers.pkl"), "rb") as f:
        return pickle.load(f)


In [ ]:
%%writefile src/evaluate.py
"""
Metrics, reporting, and the master results log.
Always report macro F1 + per-class breakdown — never accuracy alone.
"""
import json
import os

import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score

CLASSES = [0, 1, 2, 3, 4, 5, 6]
RESULTS_LOG = "experiments/results.csv"

# Columns shown in --compare (focused on rare/hard classes)
_COMPARE_COLS = [
    "experiment", "macro_f1",
    "f1_c0", "f1_c1", "f1_c3", "f1_c4", "f1_c5", "f1_c6",
    "duration_s", "timestamp", "notes",
]


def compute_metrics(y_true, y_pred) -> dict:
    macro_f1 = f1_score(y_true, y_pred, average="macro", labels=CLASSES, zero_division=0)
    report = classification_report(
        y_true, y_pred, labels=CLASSES, zero_division=0, output_dict=True
    )
    cm = confusion_matrix(y_true, y_pred, labels=CLASSES)
    return {"macro_f1": macro_f1, "report": report, "confusion_matrix": cm.tolist()}


def print_metrics(metrics: dict, title: str = "") -> None:
    if title:
        print(f"\n{'='*55}\n{title}\n{'='*55}")
    print(f"Macro F1: {metrics['macro_f1']:.4f}\n")
    print(f"{'Class':<8} {'F1':>6} {'Prec':>6} {'Rec':>6} {'N':>6}")
    print("-" * 38)
    for cls in CLASSES:
        r = metrics["report"].get(str(cls), {})
        f1  = r.get("f1-score",  0.0)
        pre = r.get("precision", 0.0)
        rec = r.get("recall",    0.0)
        n   = int(r.get("support", 0))
        print(f"{cls:<8} {f1:>6.3f} {pre:>6.3f} {rec:>6.3f} {n:>6}")
    print()


def save_metrics(metrics: dict, out_dir: str) -> None:
    if out_dir.endswith(".json"):
        path = out_dir
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    else:
        os.makedirs(out_dir, exist_ok=True)
        path = os.path.join(out_dir, "metrics.json")
    with open(path, "w") as f:
        json.dump(metrics, f, indent=2)


def append_to_results_log(
    exp_name: str,
    metrics: dict,
    config: dict,
    timestamp: str = "",
    duration: float = 0.0,
    notes: str = "",
) -> None:
    """Upsert one row into experiments/results.csv."""
    row = {
        "experiment":  exp_name,
        "macro_f1":    round(metrics["macro_f1"], 4),
        "model_type":  config.get("model", {}).get("type", ""),
        "n_folds":     config.get("data", {}).get("n_folds", ""),
        "seed":        config.get("seed", ""),
        "timestamp":   timestamp,
        "duration_s":  round(duration, 1),
        "notes":       notes or config.get("notes", ""),
    }
    for cls in CLASSES:
        r = metrics["report"].get(str(cls), {})
        row[f"f1_c{cls}"] = round(r.get("f1-score", 0.0), 4)

    df_row = pd.DataFrame([row])
    if os.path.exists(RESULTS_LOG):
        existing = pd.read_csv(RESULTS_LOG, keep_default_na=False)
        existing = existing[existing["experiment"] != exp_name]
        df_out = pd.concat([existing, df_row], ignore_index=True)
    else:
        os.makedirs(os.path.dirname(RESULTS_LOG), exist_ok=True)
        df_out = df_row

    df_out = df_out.sort_values("macro_f1", ascending=False)
    df_out.to_csv(RESULTS_LOG, index=False)
    print(f"Results logged → {RESULTS_LOG}")


def print_comparison(full: bool = False) -> None:
    """Print the leaderboard table. full=True shows all columns."""
    if not os.path.exists(RESULTS_LOG):
        print("No experiments logged yet.")
        return

    df = (pd.read_csv(RESULTS_LOG, keep_default_na=False)
            .sort_values("macro_f1", ascending=False)
            .reset_index(drop=True))
    df.index += 1  # rank starts at 1

    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 140)
    pd.set_option("display.float_format", "{:.4f}".format)

    print(f"\n{'='*55}")
    print(f"  Experiment leaderboard  (OOF macro F1, descending)")
    print(f"{'='*55}")

    if full:
        print(df.to_string())
    else:
        cols = [c for c in _COMPARE_COLS if c in df.columns]
        print(df[cols].to_string())

    # Highlight the improvement vs the row below
    if len(df) > 1:
        best = df["macro_f1"].iloc[0]
        second = df["macro_f1"].iloc[1]
        print(f"\n  Best vs 2nd: +{best - second:+.4f}  ({df['experiment'].iloc[0]})")
    print()


In [ ]:
%%writefile src/threshold.py
"""
Post-processing: per-class decision threshold tuning on OOF probabilities.
Tuning is explicitly allowed by competition rules. Always tune on OOF only.
"""
import numpy as np
from sklearn.metrics import f1_score

CLASSES = [0, 1, 2, 3, 4, 5, 6]


def tune_thresholds(
    oof_probs: np.ndarray,
    oof_labels: np.ndarray,
    n_iter: int = 500,
    seed: int = 42,
) -> np.ndarray:
    """
    Random-search for per-class additive offsets that maximise OOF macro F1.
    Returns an offset array of shape (n_classes,).
    Apply at inference time: preds = argmax(probs + offsets, axis=1).
    """
    baseline_preds = np.argmax(oof_probs, axis=1)
    best_f1 = f1_score(
        oof_labels, baseline_preds, average="macro", labels=CLASSES, zero_division=0
    )
    best_offsets = np.zeros(len(CLASSES))
    print(f"Threshold tuning baseline macro F1: {best_f1:.4f}")

    rng = np.random.RandomState(seed)
    for _ in range(n_iter):
        offsets = rng.uniform(-0.3, 0.3, size=len(CLASSES))
        preds = np.argmax(oof_probs + offsets, axis=1)
        f1 = f1_score(oof_labels, preds, average="macro", labels=CLASSES, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_offsets = offsets.copy()

    print(f"Threshold tuning best macro F1:     {best_f1:.4f}")
    print(f"Offsets: {dict(zip(CLASSES, best_offsets.round(4)))}")
    return best_offsets


def apply_thresholds(probs: np.ndarray, offsets: np.ndarray) -> np.ndarray:
    return np.argmax(probs + offsets, axis=1)


In [ ]:
%%writefile src/logging_utils.py
"""
Dual-output logging: every print() goes to both the terminal and a log file.
tqdm writes to stderr so it stays on the terminal only — logs stay clean.
"""
import io
import os
import re
import sys
from contextlib import contextmanager
from datetime import datetime


# Strip ANSI escape codes (colour, cursor movement) before writing to file
_ANSI_RE = re.compile(r"\x1b\[[0-9;]*[mABCDEFGHJKSTfhilmnsu]|\r")


class _Tee(io.TextIOBase):
    """Writes to both the original stdout and a log file simultaneously."""

    def __init__(self, log_path: str, original_stdout):
        self._stdout = original_stdout
        self._file = open(log_path, "a", encoding="utf-8", buffering=1)

    def write(self, text: str) -> int:
        self._stdout.write(text)
        self._file.write(_ANSI_RE.sub("", text))
        return len(text)

    def flush(self):
        self._stdout.flush()
        self._file.flush()

    def close(self):
        self._file.close()

    # Needed so tqdm and other libs can check isatty on the underlying stream
    def isatty(self) -> bool:
        return self._stdout.isatty()

    @property
    def encoding(self):
        return self._stdout.encoding


@contextmanager
def run_log(out_dir: str, filename: str = "train.log"):
    """
    Context manager that tees all print() output to out_dir/train.log.
    Usage:
        with run_log(out_dir):
            ... training code ...
    """
    os.makedirs(out_dir, exist_ok=True)
    log_path = os.path.join(out_dir, filename)

    tee = _Tee(log_path, sys.stdout)
    original_stdout = sys.stdout
    sys.stdout = tee
    try:
        print(f"[log] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}  →  {log_path}")
        yield log_path
    finally:
        sys.stdout = original_stdout
        tee.close()


In [ ]:
%%writefile src/models/__init__.py
"""
Model dispatcher — routes build/save/load to the right backend by model type.
Add new model families here; train.py and predict.py import only from src.models.
"""
import os
import pickle

LINEAR_TYPES = {"logistic_regression", "linear_svc"}
GBM_TYPES = {"lgbm"}


def build_model(config: dict):
    model_type = config.get("type", "logistic_regression")
    if model_type in LINEAR_TYPES:
        from src.models.linear import build_model as _build
        return _build(config)
    if model_type in GBM_TYPES:
        from src.models.gbm import build_model as _build
        return _build(config)
    raise ValueError(
        f"Unknown model type: '{model_type}'. "
        f"Choose from: {sorted(LINEAR_TYPES | GBM_TYPES)}"
    )


def save_model(model, out_dir: str) -> None:
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, "model.pkl"), "wb") as f:
        pickle.dump(model, f)


def load_model(out_dir: str):
    with open(os.path.join(out_dir, "model.pkl"), "rb") as f:
        return pickle.load(f)


In [ ]:
%%writefile src/models/transformer.py
"""
HuggingFace transformer model helpers — build / save / load.
Actual training is handled by src/train_transformer.py.
"""
import os

TRANSFORMER_TYPES = {"transformer", "bert", "xlmr"}
N_CLASSES = 7


def build_model(config: dict):
    """Return (model, tokenizer) initialised from a pretrained checkpoint."""
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    pretrained = config.get("pretrained", "neuralmind/bert-base-portuguese-cased")
    tokenizer = AutoTokenizer.from_pretrained(pretrained)
    model = AutoModelForSequenceClassification.from_pretrained(
        pretrained,
        num_labels=N_CLASSES,
        ignore_mismatched_sizes=True,
    )
    return model, tokenizer


def save_model(model, tokenizer, out_dir: str) -> None:
    model_dir = os.path.join(out_dir, "model")
    os.makedirs(model_dir, exist_ok=True)
    # Unwrap DataParallel before saving
    m = model.module if hasattr(model, "module") else model
    m.save_pretrained(model_dir)
    tokenizer.save_pretrained(model_dir)


def load_model(out_dir: str):
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    model_dir = os.path.join(out_dir, "model")
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    return model, tokenizer


In [ ]:
%%writefile src/train_transformer.py
"""
GPU-accelerated transformer fine-tuning with OOF cross-validation.
Supports BERT-style models via HuggingFace transformers.

Features:
  - Mixed precision (torch.amp FP16)
  - Multi-GPU via DataParallel (uses all visible GPUs automatically)
  - Gradient accumulation
  - Linear warmup + linear decay LR schedule
  - Class-weighted cross-entropy (handles the 87% class-2 imbalance)
  - Best-epoch checkpointing within each fold
  - Weights & Biases tracking (optional — set wandb.enabled: true in config)
  - Same output structure as train.py (oof_preds.csv, metrics.json, etc.)
"""
import gc
import os
import shutil
import subprocess
import time
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset, Subset
from transformers import get_linear_schedule_with_warmup
import yaml
from tqdm import tqdm

from src.data import load_train, make_folds
from src.evaluate import append_to_results_log, compute_metrics, print_metrics, save_metrics
from src.features import clean_text
from src.logging_utils import run_log
from src.models.transformer import build_model, save_model
from src.threshold import tune_thresholds

CLASSES = [0, 1, 2, 3, 4, 5, 6]
N_CLASSES = len(CLASSES)


class _FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0, weight=None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction="none")
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()


# ── Device setup ─────────────────────────────────────────

def _setup_device(gpu_cfg=None):
    """
    Resolve which GPU(s) to use from the top-level 'gpu' config field.

    Config examples:
      gpu: all      → all available GPUs (DataParallel)
      gpu: 0        → GPU 0 only
      gpu: 1        → GPU 1 only
      gpu: [0, 1]   → both GPUs explicitly

    You can also override at the shell level with CUDA_VISIBLE_DEVICES:
      CUDA_VISIBLE_DEVICES=1 python run.py configs/exp004.yaml --train
    """
    if not torch.cuda.is_available():
        print("No GPU available — running on CPU")
        return torch.device("cpu"), []

    n_available = torch.cuda.device_count()

    if gpu_cfg is None or gpu_cfg == "all":
        gpu_ids = list(range(n_available))
    elif isinstance(gpu_cfg, int):
        gpu_ids = [gpu_cfg]
    elif isinstance(gpu_cfg, list):
        gpu_ids = [int(g) for g in gpu_cfg]
    else:
        raise ValueError(
            f"Invalid 'gpu' config value: {gpu_cfg!r}. "
            "Use an integer, a list of integers, or 'all'."
        )

    for g in gpu_ids:
        if g >= n_available:
            raise ValueError(
                f"GPU {g} requested but only {n_available} GPU(s) available "
                f"(indices 0–{n_available - 1})."
            )

    label = " + ".join(
        f"[{g}] {torch.cuda.get_device_name(g)}" for g in gpu_ids
    )
    print(f"GPUs : {label}")
    return torch.device(f"cuda:{gpu_ids[0]}"), gpu_ids


# ── Weights & Biases ──────────────────────────────────────

def _init_wandb(config, exp_name):
    """
    Initialise a wandb run and return it, or return None if wandb is
    disabled in config or not installed.
    """
    wb = config.get("wandb", {})
    if not wb.get("enabled", False):
        return None
    try:
        import wandb
        mdl = config["model"]
        run = wandb.init(
            project=wb.get("project", "spr-2026-mammography"),
            entity=wb.get("entity") or None,
            name=exp_name,
            tags=wb.get("tags", []),
            config={
                "experiment":   exp_name,
                "pretrained":   mdl["pretrained"],
                "max_length":   mdl.get("max_length", 512),
                "n_folds":      config["data"]["n_folds"],
                "seed":         config.get("seed", 42),
                **mdl.get("params", {}),
            },
            reinit=True,
        )
        # Custom x-axis so per-fold/epoch charts align correctly
        wandb.define_metric("epoch")
        wandb.define_metric("train/*", step_metric="epoch")
        wandb.define_metric("val/*",   step_metric="epoch")
        print(f"wandb run : {run.url}")
        return run
    except Exception as exc:
        print(f"wandb init failed ({exc}) — continuing without tracking")
        return None


def _wandb_log_epoch(run, fold, epoch, train_loss, ep_metrics, lr):
    if run is None:
        return
    import wandb
    log = {
        "epoch":      epoch,
        "fold":       fold,
        "train/loss": train_loss,
        "val/macro_f1": ep_metrics["macro_f1"],
        "lr":         lr,
    }
    for cls in CLASSES:
        r = ep_metrics["report"].get(str(cls), {})
        log[f"val/f1_c{cls}"]   = r.get("f1-score",  0.0)
        log[f"val/prec_c{cls}"] = r.get("precision", 0.0)
        log[f"val/rec_c{cls}"]  = r.get("recall",    0.0)
    run.log(log)


def _wandb_log_oof(run, oof_metrics, y_true, oof_preds, fold_f1s):
    if run is None:
        return
    import wandb
    # Summary scalars
    run.summary["oof/macro_f1"] = oof_metrics["macro_f1"]
    run.summary["oof/fold_f1_mean"] = float(np.mean(fold_f1s))
    run.summary["oof/fold_f1_std"]  = float(np.std(fold_f1s))
    for cls in CLASSES:
        r = oof_metrics["report"].get(str(cls), {})
        run.summary[f"oof/f1_c{cls}"] = r.get("f1-score", 0.0)

    # Confusion matrix
    run.log({"oof/confusion_matrix": wandb.plot.confusion_matrix(
        probs=None,
        y_true=y_true.tolist(),
        preds=oof_preds.tolist(),
        class_names=[f"BI-RADS {c}" for c in CLASSES],
    )})

    # Per-class F1 bar chart
    table = wandb.Table(
        columns=["class", "f1", "precision", "recall", "support"],
        data=[
            [
                f"BI-RADS {cls}",
                oof_metrics["report"].get(str(cls), {}).get("f1-score",  0.0),
                oof_metrics["report"].get(str(cls), {}).get("precision", 0.0),
                oof_metrics["report"].get(str(cls), {}).get("recall",    0.0),
                int(oof_metrics["report"].get(str(cls), {}).get("support", 0)),
            ]
            for cls in CLASSES
        ],
    )
    run.log({"oof/per_class": wandb.plot.bar(table, "class", "f1", title="OOF F1 per class")})


# ── Dataset ───────────────────────────────────────────────

class _TextDataset(Dataset):
    """Tokenizes all texts upfront and caches tensors in memory."""

    def __init__(self, texts, tokenizer, max_length, labels=None):
        enc = tokenizer(
            texts,
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        self.input_ids      = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.token_type_ids = enc.get("token_type_ids")
        self.labels = (
            torch.tensor(labels, dtype=torch.long) if labels is not None else None
        )

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        item = {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
        }
        if self.token_type_ids is not None:
            item["token_type_ids"] = self.token_type_ids[idx]
        if self.labels is not None:
            item["labels"] = self.labels[idx]
        return item


# ── Class weights ─────────────────────────────────────────

def _class_weights(y):
    counts = np.bincount(y, minlength=N_CLASSES).astype(float)
    total  = counts.sum()
    w = np.zeros(N_CLASSES)
    present = counts > 0
    w[present] = total / (N_CLASSES * counts[present])
    return torch.tensor(w, dtype=torch.float32)


# ── Training helpers ──────────────────────────────────────

def _train_epoch(model, loader, optimizer, scaler, scheduler, criterion, device, grad_accum):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(loader):
        ids   = batch["input_ids"].to(device, non_blocking=True)
        mask  = batch["attention_mask"].to(device, non_blocking=True)
        ttids = batch.get("token_type_ids")
        if ttids is not None:
            ttids = ttids.to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        with autocast(device_type="cuda", enabled=scaler.is_enabled()):
            kwargs = {"input_ids": ids, "attention_mask": mask}
            if ttids is not None:
                kwargs["token_type_ids"] = ttids
            out    = model(**kwargs)
            logits = out.logits if hasattr(out, "logits") else out
            loss   = criterion(logits.float(), labels) / grad_accum

        scaler.scale(loss).backward()
        total_loss += loss.item() * grad_accum

        if (step + 1) % grad_accum == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            params = (model.module if hasattr(model, "module") else model).parameters()
            nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

    return total_loss / len(loader)


@torch.no_grad()
def _predict(model, loader, device, use_amp):
    model.eval()
    all_probs = []
    for batch in loader:
        ids   = batch["input_ids"].to(device, non_blocking=True)
        mask  = batch["attention_mask"].to(device, non_blocking=True)
        ttids = batch.get("token_type_ids")
        if ttids is not None:
            ttids = ttids.to(device, non_blocking=True)

        with autocast(device_type="cuda", enabled=use_amp):
            kwargs = {"input_ids": ids, "attention_mask": mask}
            if ttids is not None:
                kwargs["token_type_ids"] = ttids
            out    = model(**kwargs)
            logits = out.logits if hasattr(out, "logits") else out

        probs = torch.softmax(logits.float(), dim=-1).cpu().numpy()
        all_probs.append(probs)

    return np.concatenate(all_probs, axis=0)


# ── Entry point ───────────────────────────────────────────

def run_training_transformer(config_path: str, notes_override: str = None) -> str:
    with open(config_path) as f:
        config = yaml.safe_load(f)

    exp_name = config["experiment_name"]
    out_dir  = os.path.join("experiments", exp_name)
    os.makedirs(out_dir, exist_ok=True)

    dest = os.path.join(out_dir, "config.yaml")
    if os.path.abspath(config_path) != os.path.abspath(dest):
        shutil.copy(config_path, dest)

    notes = notes_override or config.get("notes", "")

    with run_log(out_dir):
        _run(config, exp_name, out_dir, config_path, notes)

    return out_dir


def _git_hash():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL
        ).decode().strip()
    except Exception:
        return "unknown"


def _run(config, exp_name, out_dir, config_path, notes):
    t_start   = time.time()
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    seed       = config.get("seed", 42)
    n_folds    = config["data"]["n_folds"]
    mdl_cfg    = config["model"]
    pretrained      = mdl_cfg["pretrained"]
    max_length      = mdl_cfg.get("max_length", 512)
    p               = mdl_cfg.get("params", {})
    batch_size      = p.get("batch_size", 32)
    eval_batch_size = p.get("eval_batch_size", batch_size * 2)
    grad_accum      = p.get("gradient_accumulation_steps", 1)
    n_epochs        = p.get("epochs", 5)
    lr              = p.get("learning_rate", 2e-5)
    warmup_ratio    = p.get("warmup_ratio", 0.1)
    weight_decay    = p.get("weight_decay", 0.01)
    fp16            = p.get("fp16", True)
    loss_fn                  = p.get("loss_fn", "cross_entropy")   # "cross_entropy" | "focal"
    focal_gamma              = p.get("focal_gamma", 2.0)
    early_stopping_patience  = p.get("early_stopping_patience", None)  # None = disabled
    label_smoothing          = p.get("label_smoothing", 0.0)           # 0.0 = disabled

    gpu_cfg = config.get("gpu", "all")

    torch.manual_seed(seed)
    np.random.seed(seed)

    print(f"\n{'='*55}")
    print(f"Experiment : {exp_name}")
    print(f"Config     : {config_path}")
    print(f"Started    : {timestamp}")
    print(f"Git commit : {_git_hash()}")
    print(f"Pretrained : {pretrained}")
    if notes:
        print(f"Notes      : {notes}")
    print(f"{'='*55}")

    device, gpu_ids = _setup_device(gpu_cfg)
    use_amp = fp16 and torch.cuda.is_available()

    wb_run = _init_wandb(config, exp_name)

    # ── Load data ─────────────────────────────────────────
    print("Loading data...", end=" ", flush=True)
    df = load_train()
    df["text"] = df["report"].apply(clean_text)
    df = make_folds(df, n_folds=n_folds, seed=seed)
    print(f"done  ({len(df):,} rows, {n_folds} folds)")

    cw = _class_weights(df["target"].values).to(device)
    if loss_fn == "focal":
        criterion = _FocalLoss(gamma=focal_gamma, weight=cw)
    else:
        criterion = nn.CrossEntropyLoss(weight=cw, label_smoothing=label_smoothing)

    # ── Tokenize once upfront ────────────────────────────
    print(f"Tokenizing with '{pretrained}'...", end=" ", flush=True)
    _, tokenizer = build_model({"pretrained": pretrained})
    full_ds = _TextDataset(
        df["text"].tolist(), tokenizer, max_length, labels=df["target"].values
    )
    print("done")

    oof_probs = np.zeros((len(df), N_CLASSES))
    fold_f1s  = []
    global_epoch = 0  # monotonically increasing across all folds for wandb x-axis

    # ── OOF cross-validation ──────────────────────────────
    fold_bar = tqdm(range(n_folds), desc="CV folds", unit="fold", ncols=72)
    for fold in fold_bar:
        train_pos = df.index[df["fold"] != fold].tolist()
        val_pos   = df.index[df["fold"] == fold].tolist()

        train_ds = Subset(full_ds, train_pos)
        val_ds   = Subset(full_ds, val_pos)

        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                                  num_workers=4, pin_memory=True)
        val_loader   = DataLoader(val_ds, batch_size=eval_batch_size, shuffle=False,
                                  num_workers=4, pin_memory=True)

        model, _ = build_model({"pretrained": pretrained})
        model = model.to(device)
        if len(gpu_ids) > 1:
            model = nn.DataParallel(model, device_ids=gpu_ids)

        steps_per_epoch = max(1, len(train_loader) // grad_accum)
        total_steps  = steps_per_epoch * n_epochs
        warmup_steps = max(1, int(total_steps * warmup_ratio))

        base = model.module if hasattr(model, "module") else model
        optimizer = torch.optim.AdamW(base.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = get_linear_schedule_with_warmup(
            optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
        )
        scaler = GradScaler(enabled=use_amp)

        best_f1    = -1.0
        best_probs = None
        no_improve = 0

        for epoch in range(n_epochs):
            global_epoch += 1
            train_loss = _train_epoch(
                model, train_loader, optimizer, scaler, scheduler,
                criterion, device, grad_accum,
            )
            val_probs  = _predict(model, val_loader, device, use_amp)
            y_val      = df.loc[val_pos, "target"].values
            ep_metrics = compute_metrics(y_val, val_probs.argmax(1))
            ep_f1      = ep_metrics["macro_f1"]
            current_lr = scheduler.get_last_lr()[0]

            fold_bar.set_postfix({
                "fold": f"{fold+1}/{n_folds}",
                "ep":   f"{epoch+1}/{n_epochs}",
                "loss": f"{train_loss:.3f}",
                "F1":   f"{ep_f1:.4f}",
            }, refresh=True)

            _wandb_log_epoch(wb_run, fold + 1, global_epoch, train_loss, ep_metrics, current_lr)

            if ep_f1 > best_f1:
                best_f1    = ep_f1
                best_probs = val_probs.copy()
                no_improve = 0
            else:
                no_improve += 1
                if early_stopping_patience is not None and no_improve >= early_stopping_patience:
                    tqdm.write(f"  early stop fold {fold+1} at epoch {epoch+1} "
                               f"(no improvement for {early_stopping_patience} epochs)")
                    break

        for i, pos in enumerate(val_pos):
            oof_probs[pos] = best_probs[i]
        fold_f1s.append(best_f1)

        if wb_run:
            wb_run.log({"fold_best_f1": best_f1, "fold": fold + 1})

        del model, optimizer, scheduler, scaler
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    fold_bar.close()

    # ── OOF evaluation ────────────────────────────────────
    oof_preds   = oof_probs.argmax(axis=1)
    oof_metrics = compute_metrics(df["target"].values, oof_preds)
    print_metrics(oof_metrics, title=f"OOF Results — {exp_name}")
    print(f"Fold F1s : {[round(f, 4) for f in fold_f1s]}")
    print(f"Mean ± σ : {np.mean(fold_f1s):.4f} ± {np.std(fold_f1s):.4f}")

    oof_df = df[["ID", "target", "fold"]].copy()
    oof_df["oof_pred"] = oof_preds
    for i, cls in enumerate(CLASSES):
        oof_df[f"p{cls}"] = oof_probs[:, i]
    oof_df.to_csv(os.path.join(out_dir, "oof_preds.csv"), index=False)
    save_metrics(oof_metrics, out_dir)

    _wandb_log_oof(wb_run, oof_metrics, df["target"].values, oof_preds, fold_f1s)

    # ── Optional threshold tuning ─────────────────────────
    if config.get("threshold", {}).get("enabled", False):
        print("\n--- Threshold Tuning ---")
        offsets = tune_thresholds(oof_probs, df["target"].values, seed=seed)
        np.save(os.path.join(out_dir, "thresholds.npy"), offsets)
        tuned_preds   = oof_probs + offsets
        tuned_metrics = compute_metrics(df["target"].values, tuned_preds.argmax(1))
        print_metrics(tuned_metrics, title="OOF Results (after threshold tuning)")
        save_metrics(tuned_metrics, os.path.join(out_dir, "metrics_tuned.json"))
        oof_metrics = tuned_metrics
        if wb_run:
            wb_run.summary["oof/macro_f1_tuned"] = tuned_metrics["macro_f1"]

    # ── Retrain on full data ──────────────────────────────
    print("\n--- Retraining on full data ---")
    full_loader = DataLoader(full_ds, batch_size=batch_size, shuffle=True,
                             num_workers=4, pin_memory=True)

    model, tokenizer_full = build_model({"pretrained": pretrained})
    model = model.to(device)
    if len(gpu_ids) > 1:
        model = nn.DataParallel(model, device_ids=gpu_ids)

    total_steps  = max(1, len(full_loader) // grad_accum) * n_epochs
    warmup_steps = max(1, int(total_steps * warmup_ratio))
    base         = model.module if hasattr(model, "module") else model
    optimizer    = torch.optim.AdamW(base.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler    = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )
    scaler = GradScaler(enabled=use_amp)

    for epoch in tqdm(range(n_epochs), desc="Full retrain", unit="ep", ncols=70):
        _train_epoch(model, full_loader, optimizer, scaler, scheduler,
                     criterion, device, grad_accum)

    save_model(model, tokenizer_full, out_dir)
    print(f"Model saved → experiments/{exp_name}/model/")

    if wb_run:
        wb_run.finish()

    duration = time.time() - t_start
    print(f"\nDuration   : {duration:.0f}s  ({duration/60:.1f}m)")
    append_to_results_log(exp_name, oof_metrics, config,
                          timestamp=timestamp, duration=duration, notes=notes)
    print(f"Outputs    : experiments/{exp_name}/")


In [ ]:
%%writefile src/predict.py
"""
Generate submission.csv from a trained experiment directory.
Dispatches to sklearn or transformer predict based on what's in the experiment dir.
Called by run.py — do not invoke directly.
"""
import os

import numpy as np
import pandas as pd

from src.data import load_test
from src.features import clean_text, load_vectorizers, transform_features
from src.models import load_model

CLASSES = [0, 1, 2, 3, 4, 5, 6]


def _write_submission(test, preds, exp_dir):
    sub = pd.DataFrame({"ID": test["ID"], "target": preds})
    sub_path = os.path.join(exp_dir, "submission.csv")
    sub.to_csv(sub_path, index=False)
    print(f"Submission saved → {sub_path}  ({len(sub)} rows)")
    print(f"Prediction distribution:\n{sub['target'].value_counts().sort_index().to_string()}")
    return sub_path


def run_predict_transformer(exp_dir: str) -> str:
    import torch
    from torch.amp import autocast
    from torch.utils.data import DataLoader
    from src.models.transformer import load_model as load_transformer

    test = load_test()
    test["text"] = test["report"].apply(clean_text)

    model, tokenizer = load_transformer(exp_dir)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device).eval()

    # Read max_length from saved config if present
    import yaml
    cfg_path = os.path.join(exp_dir, "config.yaml")
    max_length = 512
    if os.path.exists(cfg_path):
        with open(cfg_path) as f:
            cfg = yaml.safe_load(f)
        max_length = cfg.get("model", {}).get("max_length", 512)

    from src.train_transformer import _TextDataset
    ds = _TextDataset(test["text"].tolist(), tokenizer, max_length)
    loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

    all_probs = []
    with torch.no_grad():
        for batch in loader:
            ids  = batch["input_ids"].to(device, non_blocking=True)
            mask = batch["attention_mask"].to(device, non_blocking=True)
            ttids = batch.get("token_type_ids")
            kwargs = {"input_ids": ids, "attention_mask": mask}
            if ttids is not None:
                kwargs["token_type_ids"] = ttids.to(device, non_blocking=True)
            with autocast(device_type="cuda", enabled=torch.cuda.is_available()):
                out = model(**kwargs)
            probs = torch.softmax(out.logits.float(), dim=-1).cpu().numpy()
            all_probs.append(probs)

    prob_matrix = np.concatenate(all_probs, axis=0)

    threshold_path = os.path.join(exp_dir, "thresholds.npy")
    if os.path.exists(threshold_path):
        offsets = np.load(threshold_path)
        preds = np.argmax(prob_matrix + offsets, axis=1)
        print("Applied saved threshold offsets.")
    else:
        preds = np.argmax(prob_matrix, axis=1)

    return _write_submission(test, preds, exp_dir)


def run_predict(exp_dir: str) -> str:
    test = load_test()
    test["text"] = test["report"].apply(clean_text)

    vectorizers = load_vectorizers(exp_dir)
    model = load_model(exp_dir)

    X_test = transform_features(test["text"].tolist(), vectorizers)
    probs = model.predict_proba(X_test)

    # Align probability columns to CLASSES order
    prob_matrix = np.zeros((len(test), len(CLASSES)))
    for i, cls in enumerate(model.classes_):
        prob_matrix[:, CLASSES.index(int(cls))] = probs[:, i]

    # Apply thresholds if they were tuned for this experiment
    threshold_path = os.path.join(exp_dir, "thresholds.npy")
    if os.path.exists(threshold_path):
        offsets = np.load(threshold_path)
        preds = np.argmax(prob_matrix + offsets, axis=1)
        print("Applied saved threshold offsets.")
    else:
        preds = np.argmax(prob_matrix, axis=1)

    return _write_submission(test, preds, exp_dir)


In [ ]:
%%writefile configs/exp010_bertimbau_es.yaml
experiment_name: exp010_bertimbau_es
notes: "BERTimbau base + early stopping \u2014 re-run of best model (exp004) with\
  \ epochs=10, patience=2, max_length=256; tests whether more epochs + ES beats 5-epoch\
  \ fixed run"
seed: 42
gpu: all
data:
  n_folds: 5
model:
  type: transformer
  pretrained: neuralmind/bert-base-portuguese-cased
  max_length: 256
  params:
    epochs: 10
    batch_size: 64
    eval_batch_size: 128
    gradient_accumulation_steps: 1
    learning_rate: 2.0e-05
    warmup_ratio: 0.1
    weight_decay: 0.01
    fp16: true
    early_stopping_patience: 2
    loss_fn: cross_entropy
threshold:
  enabled: true
wandb:
  enabled: false


In [ ]:
# ── Model path ──────────────────────────────────────────────────────
# Auto-detects the correct subdirectory containing config.json.
# Falls back to HuggingFace ID when internet is ON (local testing).
import os

_KAGGLE_MODEL_ROOT = '/kaggle/input/models/caokhoihuynh/bert-base-portuguese-cased/pytorch/default/1'
_HF_ID = 'neuralmind/bert-base-portuguese-cased'

def _find_model_path(root, hf_id):
    """Walk root to find the dir containing config.json; fall back to hf_id."""
    if os.path.isdir(root):
        for dirpath, _, files in os.walk(root):
            if 'config.json' in files:
                print(f'Found model at: {dirpath}')
                return dirpath
        print(f'WARNING: config.json not found under {root}, using HF id')
    return hf_id

PRETRAINED_PATH = _find_model_path(_KAGGLE_MODEL_ROOT, _HF_ID)
print(f'PRETRAINED_PATH = {PRETRAINED_PATH}')

import yaml
with open('configs/exp010_bertimbau_es.yaml') as f:
    config = yaml.safe_load(f)
config['model']['pretrained'] = PRETRAINED_PATH
with open('configs/exp010_bertimbau_es.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)
print('Config written →', 'configs/exp010_bertimbau_es.yaml')


In [ ]:
import os, shutil
from src.train_transformer import run_training_transformer
from src.predict import run_predict_transformer

exp_dir = run_training_transformer('configs/exp010_bertimbau_es.yaml')
run_predict_transformer(exp_dir)

shutil.copy(os.path.join(exp_dir, 'submission.csv'), 'submission.csv')
print('submission.csv ready at', os.getcwd())


In [ ]:
import pandas as pd
sub = pd.read_csv('submission.csv')
assert {'ID', 'target'}.issubset(sub.columns)
assert sub['target'].between(0, 6).all(), 'Targets out of range'
print(f'submission.csv: {len(sub)} rows')
print(sub['target'].value_counts().sort_index().to_string())
sub.head()
